In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from iblatlas.atlas import BrainRegions
from one.api import ONE
from brainbox.population.decode import get_spike_counts_in_bins
from brainbox.io.one import SpikeSortingLoader, SessionLoader
from brainbox.ephys_plots import plot_brain_regions
from brainbox.task.trials import get_event_aligned_raster, get_psth
from iblatlas.atlas import AllenAtlas
from brainbox.behavior.training import compute_performance, plot_psychometric, plot_reaction_time
from brainbox.task.trials import find_trial_ids
from brainbox.io.one import SessionLoader
from brainbox.population.decode import get_spike_counts_in_bins
from pathlib import Path
from brainbox.task.trials import get_event_aligned_raster, get_psth
from brainbox import singlecell
from tqdm.notebook import tqdm
import seaborn as sns

from iblatlas.atlas import AllenAtlas
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from brainwidemap.bwm_loading import merge_probes
from brainbox.behavior.training import compute_performance, plot_psychometric, plot_reaction_time
from brainbox.io.one import SessionLoader
from pathlib import Path
from brainbox.task.trials import get_event_aligned_raster, get_psth
from brainbox.singlecell import bin_spikes2D
import numpy as np
from iblatlas.atlas import BrainRegions
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
import itertools
import pickle as pkl
from tqdm import tqdm
from pathlib import Path
import warnings
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from ibl_info.pcaprojections import analyze_neural_interaction, analyze_inter_region_interaction


%load_ext autoreload
%autoreload 2


In [3]:
# load data and check interaction

In [4]:
# connect to server
one = ONE(
    base_url="https://openalyx.internationalbrainlab.org", password="international", silent=True
)
# select example eid for decoding analysis
subject_id = "CSH_ZAD_022"
eid = "a82800ce-f4e3-4464-9b80-4c3d6fade333"
session_id = eid

In [5]:
from ibl_info.prepare_data_pid import get_new_cinc_intervals, prepare_ephys_data


session_id = eid
pids, probes = one.eid2pid(session_id)
if isinstance(probes, list) and len(probes) > 1:
    to_merge = [load_good_units(one, pid=pid, qc=1) for pid in pids]
    spikes, clusters = merge_probes(
        [spikes for spikes, _ in to_merge], [clusters for _, clusters in to_merge]
    )
else:
    spikes, clusters = load_good_units(one, pid=pids[0], qc=1)

trials, mask = load_trials_and_mask(one, session_id, exclude_nochoice=True, exclude_unbiased=True)
trials = trials[mask]

intervals, target_variable, congruent_flags, incongruent_flags = get_new_cinc_intervals(
    trials, "stim"
)

binned_spikes, actual_regions, n_units, cluster_uuids_list = prepare_ephys_data(
    spikes, clusters, intervals, ["LGd"], minimum_units=5
)  # this returns all neurons from a single region that pass qc
# however, it is in trials x neurons

spike_data = binned_spikes[0].T

congruent_target = target_variable[congruent_flags]
incongruent_target = target_variable[incongruent_flags]

congruent_spikes = spike_data[:, congruent_flags]
incongruent_spikes = spike_data[:, incongruent_flags]

Region found LGd, 67


In [10]:
import warnings

warnings.filterwarnings("ignore")

In [25]:
binned_spikes[0].shape

(367, 67)

In [14]:
results_congruent = analyze_neural_interaction(
    neural_data=congruent_spikes.T, labels=congruent_target
)

Completed bootstrap 10/50
Completed bootstrap 20/50
Completed bootstrap 30/50
Completed bootstrap 40/50
Completed bootstrap 50/50


In [23]:
results_congruent_t = analyze_neural_interaction(
    neural_data=congruent_spikes.T, labels=congruent_target
)

(236, 33) (60, 33)


In [18]:
results_congruent_t["mean_test_accuracy"]

np.float64(0.7963762711864407)

In [19]:
results_congruent["mean_test_accuracy"]

np.float64(0.7964237288135594)

In [55]:
results = analyze_neural_interaction(
    neural_data=incongruent_spikes.T,
    labels=incongruent_target.T,
)

Completed bootstrap 10/100
Completed bootstrap 20/100
Completed bootstrap 30/100
Completed bootstrap 40/100
Completed bootstrap 50/100
Completed bootstrap 60/100
Completed bootstrap 70/100
Completed bootstrap 80/100
Completed bootstrap 90/100
Completed bootstrap 100/100


In [20]:
def plot_interaction_significance(results):
    # Extract interaction coefficients (index 2)
    interactions = results["all_bootstrap_coefficients"][:, 2]

    # Calculate 95% Confidence Interval
    ci_lower = np.percentile(interactions, 2.5)
    ci_upper = np.percentile(interactions, 97.5)
    mean_val = np.mean(interactions)

    # Check significance
    is_significant = (ci_lower > 0) or (ci_upper < 0)

    # Visualization
    plt.figure(figsize=(8, 5))
    plt.hist(interactions, bins=20, alpha=0.7, color="purple", edgecolor="black")
    plt.axvline(0, color="red", linestyle="--", linewidth=2, label="Zero")
    plt.axvline(mean_val, color="black", linewidth=2, label=f"Mean: {mean_val:.2f}")
    plt.axvline(ci_lower, color="gray", linestyle=":", label="95% CI")
    plt.axvline(ci_upper, color="gray", linestyle=":")

    plt.title(f"Interaction Significance (Significant: {is_significant})")
    plt.xlabel("Standardized Interaction Coefficient")
    plt.ylabel("Bootstrap Count")
    plt.legend()
    plt.show()

In [ ]:
def plot_coefficient_comparison(results):
    # Shape: (n_bootstraps, 3) -> [PC1, PC2, Interaction]
    coefs = results["all_bootstrap_coefficients"]

    plt.figure(figsize=(8, 6))
    plt.boxplot(
        [coefs[:, 0], coefs[:, 1], coefs[:, 2]],
        labels=["Subset 1 (Main)", "Subset 2 (Main)", "Interaction"],
    )

    plt.axhline(0, color="gray", linestyle="--")
    plt.ylabel("Standardized Log-Odds Contribution")
    plt.title("Relative Importance of Neural Terms")

    # Calculate Synergy Ratio
    # |Interaction| / (|Main1| + |Main2|)
    synergy_ratio = np.abs(coefs[:, 2]) / (np.abs(coefs[:, 0]) + np.abs(coefs[:, 1]))
    print(f"Average Synergy Ratio: {np.mean(synergy_ratio):.2f}")
    print("(>1.0 implies interaction dominates main effects)")
    plt.show()

In [56]:
results["mean_explained_variance"]

{'Subset1': np.float64(0.2959545277407563),
 'Subset2': np.float64(0.3129979128025175)}

In [58]:
results["mean_coefficients"]

{'PC1_Subset1': np.float64(-1.279629076806791),
 'PC1_Subset2': np.float64(-1.358427269762005),
 'Interaction': np.float64(0.23628029179886073)}

In [62]:
results_congruent["mean_coefficients"]

{'PC1_Subset1': np.float64(-2.018904441762338),
 'PC1_Subset2': np.float64(-1.8569814627725185),
 'Interaction': np.float64(0.4931002332251)}

In [65]:
results_congruent["mean_test_accuracy"]

np.float64(0.7961700564971753)

In [72]:
1 - congruent_target

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [77]:
results_congruent = analyze_neural_interaction(
    neural_data=congruent_spikes.T, labels=congruent_target.T
)

Completed bootstrap 10/100
Completed bootstrap 20/100
Completed bootstrap 30/100
Completed bootstrap 40/100
Completed bootstrap 50/100
Completed bootstrap 60/100
Completed bootstrap 70/100
Completed bootstrap 80/100
Completed bootstrap 90/100
Completed bootstrap 100/100


In [78]:
results_congruent["mean_coefficients"]

{'PC1_Subset1': np.float64(1.9834705581794065),
 'PC1_Subset2': np.float64(1.8809440210079302),
 'Interaction': np.float64(0.5017931705169122)}

In [83]:
results_incongruent = analyze_neural_interaction(
    neural_data=incongruent_spikes.T, labels=incongruent_target.T
)

Completed bootstrap 10/100
Completed bootstrap 20/100
Completed bootstrap 30/100
Completed bootstrap 40/100
Completed bootstrap 50/100
Completed bootstrap 60/100
Completed bootstrap 70/100
Completed bootstrap 80/100
Completed bootstrap 90/100
Completed bootstrap 100/100


In [84]:
results_incongruent["mean_coefficients"]

{'PC1_Subset1': np.float64(1.373045456290345),
 'PC1_Subset2': np.float64(1.3230429449520356),
 'Interaction': np.float64(0.2199255763190211)}

In [88]:
results_all = analyze_neural_interaction(neural_data=spike_data, labels=target_variable)

Completed bootstrap 10/100
Completed bootstrap 20/100
Completed bootstrap 30/100
Completed bootstrap 40/100
Completed bootstrap 50/100
Completed bootstrap 60/100
Completed bootstrap 70/100
Completed bootstrap 80/100
Completed bootstrap 90/100
Completed bootstrap 100/100


In [90]:
results_all["mean_coefficients"]

{'PC1_Subset1': np.float64(2.0595291236600826),
 'PC1_Subset2': np.float64(2.18980218530551),
 'Interaction': np.float64(0.4626876083153714)}